# 07b — TTS Rebuild: 3-Tier Voice Synthesis Improvement

**Problem**: XTTS-v2 output from NB07 is garbled ("grr grr gha gha") due to:
1. `repetition_penalty=10.0` (5× the config default) → audio loops/garbles
2. `temperature=0.75` too high for short text → random token sampling
3. Reference audio: 3.3s RAVDESS acted speech → bad voice conditioning
4. Using high-level `tts_api.tts()` which hides critical generation params

**Plan (3 Tiers)**:
- **Tier 1**: Fix XTTS-v2 with low-level API, proper params, LJSpeech refs (8-10s clips)
- **Tier 2**: Install & test F5-TTS (flow-matching, no autoregressive garbling)
- **Tier 3**: YourTTS (VITS-based) as fallback

**Hardware**: RTX 4060 Laptop 7.6GB VRAM | **Data**: 12,910 LJSpeech clips (22050Hz)

In [1]:
"""
Cell 1: Setup & Imports
"""
import os, sys, time, json, warnings, gc
import numpy as np
import torch
import torchaudio
import librosa
import soundfile as sf
from pathlib import Path
from IPython.display import Audio, display, HTML
import pandas as pd

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

BASE_DIR = Path("/home/nibiru/Documents/sem6project/Speech2/pipeline_v2")
DATA_DIR = BASE_DIR / "data"
TTS_TRAIN_DIR = DATA_DIR / "processed" / "tts_train"
RESULTS_DIR = BASE_DIR / "results" / "tts_rebuild"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Print GPU info
if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    free_mem = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GB | Free: {free_mem:.1f} GB")

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"TTS data: {sum(1 for _ in TTS_TRAIN_DIR.glob('*.wav'))} files in {TTS_TRAIN_DIR}")

GPU: NVIDIA GeForce RTX 4060 Laptop GPU | VRAM: 7.6 GB | Free: 7.5 GB
PyTorch: 2.9.1+cu128
Device: cuda
TTS data: 12910 files in /home/nibiru/Documents/sem6project/Speech2/pipeline_v2/data/processed/tts_train


## Cell 2: Select High-Quality Reference Audio

XTTS-v2 voice cloning quality depends heavily on reference audio:
- **Duration**: 6-15 seconds (sweet spot: 8-10s)
- **Quality**: Clean speech, no background noise
- **Domain**: Natural reading voice (LJSpeech = perfect)
- **Multiple clips**: Averaging 3-5 clips gives better speaker embedding

In [2]:
"""
Cell 2: Select & validate reference audio from LJSpeech
"""
# Load TTS manifest
tts_manifest = pd.read_csv(DATA_DIR / "metadata" / "tts_manifest.csv")
print(f"Total TTS clips: {len(tts_manifest)}")

# Filter for good reference clips: 8-10s duration (sweet spot for XTTS-v2)
ref_candidates = tts_manifest[
    (tts_manifest.duration_sec >= 8.0) & 
    (tts_manifest.duration_sec <= 10.0)
].copy()
print(f"Reference candidates (8-10s): {len(ref_candidates)}")

# Sort by duration descending, pick top 5 diverse clips
ref_candidates = ref_candidates.sort_values("duration_sec", ascending=False)
selected_refs = ref_candidates.head(5)

# Validate: load and check SNR + quality
ref_paths = []
print("\n--- Selected Reference Clips ---")
for i, (_, row) in enumerate(selected_refs.iterrows()):
    audio_path = BASE_DIR / row["audio_path"]
    y, sr = librosa.load(str(audio_path), sr=22050)
    dur = len(y) / sr
    
    # Compute SNR estimate (signal vs silence)
    rms = np.sqrt(np.mean(y**2))
    # Trim silence and recalculate
    y_trimmed, _ = librosa.effects.trim(y, top_db=30)
    rms_trimmed = np.sqrt(np.mean(y_trimmed**2))
    snr_est = 20 * np.log10(rms_trimmed / (rms - rms_trimmed + 1e-10))
    
    ref_paths.append(str(audio_path))
    print(f"  [{i}] {audio_path.name} | {dur:.1f}s | RMS: {rms_trimmed:.4f} | transcript: {row['transcript'][:60]}...")

print(f"\n✓ {len(ref_paths)} reference clips selected")

# Play the first reference to verify quality
print("\nPlaying first reference clip:")
display(Audio(ref_paths[0], rate=22050))

Total TTS clips: 12910
Reference candidates (8-10s): 3877

--- Selected Reference Clips ---
  [0] lj_002726.wav | 10.0s | RMS: 0.0973 | transcript: He told the executioner to "do it tidy," to pull it tight, a...
  [1] lj_002595.wav | 10.0s | RMS: 0.1006 | transcript: taking in preference those which are typical and invested wi...
  [2] lj_004021.wav | 10.0s | RMS: 0.0946 | transcript: Implicit confidence appears to have been placed in Watts, wh...
  [3] lj_012147.wav | 10.0s | RMS: 0.1170 | transcript: Oswald was arrested and jailed by the New Orleans Police Dep...
  [4] lj_003734.wav | 10.0s | RMS: 0.0801 | transcript: He was coachman to a gentleman at Roehampton. One day he wen...

✓ 5 reference clips selected

Playing first reference clip:


## Tier 1: Fix XTTS-v2 with Low-Level API

**Root cause fixes:**
1. Use `model.inference()` instead of `tts_api.tts()` for full parameter control
2. Lower `repetition_penalty` from 10.0 → 2.0 (config default)
3. Lower `temperature` from 0.75 → 0.3 (more deterministic)
4. Use LJSpeech 8-10s clips as reference instead of 3.3s RAVDESS
5. Use `get_conditioning_latents()` with `sound_norm_refs=True`, `librosa_trim_db=30`, `gpt_cond_len=12`
6. Grid search over key parameters

In [5]:
"""
Cell 3: Load XTTS-v2 with low-level API & compute conditioning latents
"""
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

# Load model directly (low-level)
print("Loading XTTS-v2 model...")
xtts_config = XttsConfig()
xtts_config.load_json(os.path.expanduser(
    "~/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/config.json"
))

xtts_model = Xtts.init_from_config(xtts_config)
xtts_model.load_checkpoint(
    xtts_config,
    checkpoint_dir=os.path.expanduser(
        "~/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/"
    ),
    eval=True,
    use_deepspeed=False
)
xtts_model = xtts_model.to(DEVICE)

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"✓ XTTS-v2 loaded | VRAM: {vram_used:.2f} GB")

# Compute conditioning latents from multiple LJSpeech reference clips
print(f"\nComputing conditioning latents from {len(ref_paths)} reference clips...")
print(f"  Params: gpt_cond_len=12, gpt_cond_chunk_len=4, sound_norm_refs=True")

gpt_cond_latent, speaker_embedding = xtts_model.get_conditioning_latents(
    audio_path=ref_paths,          # List of clips → averaged embedding
    gpt_cond_len=12,               # Longer conditioning = better voice capture
    gpt_cond_chunk_len=4,          # Smaller chunks for finer detail  
    sound_norm_refs=True,          # Normalize reference volume
)

print(f"✓ Conditioning latents computed")
print(f"  gpt_cond_latent shape: {gpt_cond_latent.shape}")
print(f"  speaker_embedding shape: {speaker_embedding.shape}")

Loading XTTS-v2 model...
✓ XTTS-v2 loaded | VRAM: 3.56 GB

Computing conditioning latents from 5 reference clips...
  Params: gpt_cond_len=12, gpt_cond_chunk_len=4, sound_norm_refs=True
✓ Conditioning latents computed
  gpt_cond_latent shape: torch.Size([1, 32, 1024])
  speaker_embedding shape: torch.Size([1, 512, 1])


In [6]:
"""
Cell 4: XTTS-v2 Parameter Grid Search
Test combinations of temperature × repetition_penalty to find optimal settings.
"""

test_sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Hello, how are you doing today? I hope everything is going well.",
    "The president announced major reforms to the healthcare system yesterday.",
    "She walked through the garden, admiring the beautiful flowers in bloom.",
]

# Parameter grid - focused on the most impactful params
param_grid = [
    {"temperature": 0.1, "repetition_penalty": 2.0, "top_k": 50, "top_p": 0.85},
    {"temperature": 0.2, "repetition_penalty": 2.0, "top_k": 50, "top_p": 0.85},
    {"temperature": 0.3, "repetition_penalty": 2.0, "top_k": 50, "top_p": 0.85},
    {"temperature": 0.3, "repetition_penalty": 3.0, "top_k": 50, "top_p": 0.85},
    {"temperature": 0.5, "repetition_penalty": 2.0, "top_k": 50, "top_p": 0.85},
    {"temperature": 0.65, "repetition_penalty": 2.0, "top_k": 30, "top_p": 0.80},
    # Compare with original broken settings
    {"temperature": 0.75, "repetition_penalty": 10.0, "top_k": 50, "top_p": 0.85},
]

xtts_results = []
tier1_dir = RESULTS_DIR / "tier1_xtts"
tier1_dir.mkdir(parents=True, exist_ok=True)

print(f"Running grid search: {len(param_grid)} configs × {len(test_sentences)} sentences\n")

for pi, params in enumerate(param_grid):
    config_results = {"params": params, "outputs": []}
    print(f"Config {pi}: temp={params['temperature']}, rep_pen={params['repetition_penalty']}, "
          f"top_k={params['top_k']}, top_p={params['top_p']}")
    
    for si, sentence in enumerate(test_sentences):
        t0 = time.time()
        try:
            with torch.no_grad():
                out = xtts_model.inference(
                    text=sentence,
                    language="en",
                    gpt_cond_latent=gpt_cond_latent,
                    speaker_embedding=speaker_embedding,
                    temperature=params["temperature"],
                    repetition_penalty=params["repetition_penalty"],
                    top_k=params["top_k"],
                    top_p=params["top_p"],
                    speed=1.0,
                    enable_text_splitting=True,
                    length_penalty=1.0,
                )
            
            wav = out["wav"]
            elapsed = time.time() - t0
            dur = len(wav) / 24000  # XTTS outputs at 24kHz
            
            # Quality metrics
            rms = np.sqrt(np.mean(wav**2))
            expected_dur = len(sentence.split()) * 0.35  # ~0.35s per word
            dur_ratio = dur / expected_dur if expected_dur > 0 else 0
            
            # Save audio
            out_path = tier1_dir / f"config{pi}_sent{si}.wav"
            sf.write(str(out_path), wav, 24000)
            
            config_results["outputs"].append({
                "sentence": sentence,
                "duration": dur,
                "expected_dur": expected_dur,
                "dur_ratio": dur_ratio,
                "rms": float(rms),
                "inference_time": elapsed,
                "path": str(out_path),
                "success": True,
            })
            
            status = "✓" if 0.5 < dur_ratio < 2.5 else "⚠️"
            print(f"  [{si}] {status} dur={dur:.1f}s (expect ~{expected_dur:.1f}s, ratio={dur_ratio:.2f}) "
                  f"| RMS={rms:.4f} | {elapsed:.2f}s")
            
        except Exception as e:
            elapsed = time.time() - t0
            config_results["outputs"].append({
                "sentence": sentence, "success": False, "error": str(e)
            })
            print(f"  [{si}] ✗ ERROR: {e}")
        
        torch.cuda.empty_cache()
    
    xtts_results.append(config_results)
    print()

# Summary table
print("=" * 80)
print("GRID SEARCH SUMMARY")
print("=" * 80)
print(f"{'Config':<8} {'Temp':<6} {'RepPen':<8} {'Avg Dur Ratio':<15} {'Avg RMS':<10} {'Avg Time':<10} {'Quality'}")
print("-" * 80)
for pi, cr in enumerate(xtts_results):
    successful = [o for o in cr["outputs"] if o.get("success")]
    if successful:
        avg_ratio = np.mean([o["dur_ratio"] for o in successful])
        avg_rms = np.mean([o["rms"] for o in successful])
        avg_time = np.mean([o["inference_time"] for o in successful])
        # Quality score: penalize extreme duration ratios and low RMS
        quality = "GOOD" if (0.7 < avg_ratio < 2.0 and avg_rms > 0.01) else "BAD"
        print(f"  {pi:<6} {cr['params']['temperature']:<6} {cr['params']['repetition_penalty']:<8} "
              f"{avg_ratio:<15.2f} {avg_rms:<10.4f} {avg_time:<10.2f} {quality}")
    else:
        print(f"  {pi:<6} {cr['params']['temperature']:<6} {cr['params']['repetition_penalty']:<8} FAILED")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Running grid search: 7 configs × 4 sentences

Config 0: temp=0.1, rep_pen=2.0, top_k=50, top_p=0.85
  [0] ✓ dur=7.6s (expect ~3.1s, ratio=2.40) | RMS=0.0678 | 1.54s
  [1] ⚠️ dur=27.9s (expect ~4.2s, ratio=6.65) | RMS=0.0503 | 6.12s
  [2] ⚠️ dur=27.9s (expect ~3.5s, ratio=7.98) | RMS=0.0504 | 6.12s
  [3] ⚠️ dur=24.7s (expect ~3.8s, ratio=6.42) | RMS=0.0812 | 5.31s

Config 1: temp=0.2, rep_pen=2.0, top_k=50, top_p=0.85
  [0] ⚠️ dur=27.9s (expect ~3.1s, ratio=8.87) | RMS=0.0607 | 6.12s
  [1] ⚠️ dur=18.9s (expect ~4.2s, ratio=4.50) | RMS=0.0902 | 3.92s
  [2] ⚠️ dur=18.2s (expect ~3.5s, ratio=5.19) | RMS=0.0973 | 3.75s
  [3] ⚠️ dur=10.2s (expect ~3.8s, ratio=2.64) | RMS=0.0708 | 2.00s

Config 2: temp=0.3, rep_pen=2.0, top_k=50, top_p=0.85
  [0] ⚠️ dur=27.0s (expect ~3.1s, ratio=8.56) | RMS=0.0729 | 5.87s
  [1] ⚠️ dur=27.2s (expect ~4.2s, ratio=6.47) | RMS=0.0679 | 5.92s
  [2] ⚠️ dur=24.1s (expect ~3.5s, ratio=6.87) | RMS=0.0743 | 5.16s
  [3] ⚠️ dur=15.6s (expect ~3.8s, ratio=4.04) | RMS=0.0

In [7]:
"""
Cell 5: Listen to best XTTS-v2 config outputs & select winner
"""
# Find the best config based on duration ratio closest to 1.0 and good RMS
best_config_idx = None
best_score = float("inf")

for pi, cr in enumerate(xtts_results):
    successful = [o for o in cr["outputs"] if o.get("success")]
    if not successful:
        continue
    avg_ratio = np.mean([o["dur_ratio"] for o in successful])
    avg_rms = np.mean([o["rms"] for o in successful])
    # Score: distance from ideal ratio (1.0) + penalty for low volume
    score = abs(avg_ratio - 1.0) + max(0, 0.02 - avg_rms) * 10
    if score < best_score:
        best_score = score
        best_config_idx = pi

print(f"🏆 Best XTTS-v2 config: #{best_config_idx}")
best = xtts_results[best_config_idx]
print(f"   Params: {best['params']}")

# Play all outputs from best config
for o in best["outputs"]:
    if o.get("success"):
        print(f"\n📝 \"{o['sentence'][:60]}...\"")
        print(f"   Duration: {o['duration']:.1f}s | RMS: {o['rms']:.4f}")
        display(Audio(o["path"], rate=24000))

# Also play the broken config (last one) for comparison
print("\n" + "="*60)
print("❌ BROKEN config (original settings: temp=0.75, rep_pen=10.0) for comparison:")
broken = xtts_results[-1]
for o in broken["outputs"][:2]:  # Just play first 2
    if o.get("success"):
        print(f"\n📝 \"{o['sentence'][:60]}...\"")
        print(f"   Duration: {o['duration']:.1f}s | RMS: {o['rms']:.4f}")
        display(Audio(o["path"], rate=24000))

# Save best XTTS config
best_xtts_params = best["params"]
print(f"\n✓ Best XTTS-v2 params saved: {best_xtts_params}")

🏆 Best XTTS-v2 config: #4
   Params: {'temperature': 0.5, 'repetition_penalty': 2.0, 'top_k': 50, 'top_p': 0.85}

📝 "The quick brown fox jumps over the lazy dog...."
   Duration: 8.6s | RMS: 0.0855



📝 "Hello, how are you doing today? I hope everything is going w..."
   Duration: 9.4s | RMS: 0.0984



📝 "The president announced major reforms to the healthcare syst..."
   Duration: 22.0s | RMS: 0.0687



📝 "She walked through the garden, admiring the beautiful flower..."
   Duration: 19.4s | RMS: 0.0804



❌ BROKEN config (original settings: temp=0.75, rep_pen=10.0) for comparison:

📝 "The quick brown fox jumps over the lazy dog...."
   Duration: 20.9s | RMS: 0.0844



📝 "Hello, how are you doing today? I hope everything is going w..."
   Duration: 19.5s | RMS: 0.0761



✓ Best XTTS-v2 params saved: {'temperature': 0.5, 'repetition_penalty': 2.0, 'top_k': 50, 'top_p': 0.85}


## Tier 2: F5-TTS (Flow-Matching DiT)

**Why F5-TTS?**
- Non-autoregressive → **no garbling/looping by design**
- Flow-matching diffusion transformer (~330M params)
- Zero-shot voice cloning from 5-15s reference
- Natural prosody, ~4GB VRAM
- State-of-the-art quality for single-speaker synthesis

In [3]:
"""
Cell 6: Install & Load F5-TTS
"""
import subprocess, importlib

# Define test sentences (also used in Tier 1)
test_sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Hello, how are you doing today? I hope everything is going well.",
    "The president announced major reforms to the healthcare system yesterday.",
    "She walked through the garden, admiring the beautiful flowers in bloom.",
]

# Install F5-TTS if not present
try:
    import f5_tts
    print(f"F5-TTS already installed")
except ImportError:
    print("Installing F5-TTS...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "f5-tts", "-q"])
    import f5_tts
    print(f"✓ F5-TTS installed")

# Unload XTTS-v2 to free VRAM (if loaded)
print("Freeing VRAM (unloading XTTS-v2 if loaded)...")
try:
    del xtts_model
except NameError:
    pass
torch.cuda.empty_cache()
import gc; gc.collect()
free_mem = torch.cuda.mem_get_info()[0] / 1024**3
print(f"✓ VRAM freed: {free_mem:.1f} GB available")

# Load F5-TTS
from f5_tts.api import F5TTS

print("\nLoading F5-TTS model...")
f5_model = F5TTS(device=DEVICE)

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"✓ F5-TTS loaded | VRAM: {vram_used:.2f} GB")

F5-TTS already installed
Freeing VRAM (unloading XTTS-v2 if loaded)...
✓ VRAM freed: 7.5 GB available


2026-03-03 18:37:32.552283: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-03 18:37:32.576684: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-03 18:37:33.339470: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao


Loading F5-TTS model...
Download Vocos from huggingface charactr/vocos-mel-24khz


F5TTS_v1_Base/model_1250000.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]


vocab :  /home/nibiru/.conda/envs/ml_env/lib/python3.11/site-packages/f5_tts/infer/examples/vocab.txt
token :  custom
model :  /home/nibiru/.cache/huggingface/hub/models--SWivid--F5-TTS/snapshots/84e5a410d9cead4de2f847e7c9369a6440bdfaca/F5TTS_v1_Base/model_1250000.safetensors 

✓ F5-TTS loaded | VRAM: 0.69 GB


In [4]:
"""
Cell 7: F5-TTS Synthesis Test
"""
# F5-TTS needs a reference audio + its transcript for voice cloning
# Pick the best LJSpeech reference clip and get its transcript
ref_row = tts_manifest[tts_manifest.audio_path.str.contains(Path(ref_paths[0]).name.replace('.wav',''))]
if len(ref_row) == 0:
    # Fallback: use index-based lookup
    ref_row = tts_manifest.iloc[0:1]
ref_transcript = ref_row.iloc[0]["transcript"]
ref_audio = ref_paths[0]
print(f"Reference: {Path(ref_audio).name}")
print(f"Reference transcript: {ref_transcript[:80]}...")

tier2_dir = RESULTS_DIR / "tier2_f5tts"
tier2_dir.mkdir(parents=True, exist_ok=True)

f5_results = []

print(f"\nSynthesizing {len(test_sentences)} sentences with F5-TTS...\n")

for si, sentence in enumerate(test_sentences):
    t0 = time.time()
    try:
        wav, sr, _ = f5_model.infer(
            ref_file=ref_audio,
            ref_text=ref_transcript,
            gen_text=sentence,
        )
        elapsed = time.time() - t0
        
        # wav may be tensor or numpy
        if isinstance(wav, torch.Tensor):
            wav_np = wav.cpu().numpy()
        else:
            wav_np = np.array(wav)
        
        if wav_np.ndim > 1:
            wav_np = wav_np.squeeze()
        
        dur = len(wav_np) / sr
        rms = np.sqrt(np.mean(wav_np**2))
        expected_dur = len(sentence.split()) * 0.35
        dur_ratio = dur / expected_dur if expected_dur > 0 else 0
        
        out_path = tier2_dir / f"f5_sent{si}.wav"
        sf.write(str(out_path), wav_np, sr)
        
        f5_results.append({
            "sentence": sentence,
            "duration": dur,
            "expected_dur": expected_dur,
            "dur_ratio": dur_ratio,
            "rms": float(rms),
            "inference_time": elapsed,
            "sample_rate": sr,
            "path": str(out_path),
            "success": True,
        })
        
        status = "✓" if 0.5 < dur_ratio < 2.5 else "⚠️"
        print(f"[{si}] {status} dur={dur:.1f}s (ratio={dur_ratio:.2f}) | RMS={rms:.4f} | {elapsed:.2f}s")
        print(f"    \"{sentence[:60]}\"")
        display(Audio(wav_np, rate=sr))
        
    except Exception as e:
        elapsed = time.time() - t0
        f5_results.append({"sentence": sentence, "success": False, "error": str(e)})
        print(f"[{si}] ✗ ERROR: {e}")
    
    torch.cuda.empty_cache()

print(f"\n✓ F5-TTS: {sum(1 for r in f5_results if r.get('success'))}/{len(test_sentences)} succeeded")

Reference: lj_002726.wav
Reference transcript: He told the executioner to "do it tidy," to pull it tight, and was in a state of...

Synthesizing 4 sentences with F5-TTS...

Converting audio...
Using custom reference text...

ref_text   He told the executioner to "do it tidy," to pull it tight, and was in a state of hysterical exaltation up to the very last. 
gen_text 0 The quick brown fox jumps over the lazy dog.


Generating audio in 1 batches...


100%|██████████| 1/1 [00:02<00:00,  2.71s/it]

[0] ✓ dur=3.5s (ratio=1.10) | RMS=0.0999 | 2.82s
    "The quick brown fox jumps over the lazy dog."


Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   He told the executioner to "do it tidy," to pull it tight, and was in a state of hysterical exaltation up to the very last. 
gen_text 0 Hello, how are you doing today? I hope everything is going well.


Generating audio in 1 batches...


100%|██████████| 1/1 [00:02<00:00,  2.38s/it]

[1] ✓ dur=5.0s (ratio=1.20) | RMS=0.1226 | 2.39s
    "Hello, how are you doing today? I hope everything is going w"


... (3 outputs left out)

Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   He told the executioner to "do it tidy," to pull it tight, and was in a state of hysterical exaltation up to the very last. 
gen_text 0 She walked through the garden, admiring the beautiful flowers in bloom.


Generating audio in 1 batches...


100%|██████████| 1/1 [00:02<00:00,  2.52s/it]

[3] ✓ dur=5.6s (ratio=1.45) | RMS=0.0978 | 2.53s
    "She walked through the garden, admiring the beautiful flower"



✓ F5-TTS: 4/4 succeeded


## Tier 3: YourTTS (VITS-based, Coqui)

**Why YourTTS?**
- Already available in Coqui TTS (no extra install)
- VITS architecture = end-to-end, no autoregressive decoding
- Multi-speaker, multi-lingual
- Fast inference (~2× faster than XTTS-v2)
- Reliable fallback option

In [ ]:
"""
Cell 8: YourTTS Synthesis Test  
"""
# Free VRAM from F5-TTS
print("Freeing VRAM (unloading F5-TTS)...")
try:
    del f5_model
except NameError:
    pass
torch.cuda.empty_cache()
gc.collect()
free_mem = torch.cuda.mem_get_info()[0] / 1024**3
print(f"✓ VRAM freed: {free_mem:.1f} GB available")

# Load YourTTS via Coqui TTS high-level API
from TTS.api import TTS as CoquiTTS

yourtts_results = []
tier3_dir = RESULTS_DIR / "tier3_yourtts"
tier3_dir.mkdir(parents=True, exist_ok=True)

try:
    print("\nLoading YourTTS...")
    # Force re-download to fix corrupt zip
    import shutil
    yourtts_dir = os.path.expanduser("~/.local/share/tts/tts_models--multilingual--multi-dataset--your_tts/")
    corrupt_zip = os.path.join(yourtts_dir, "tts_models--multilingual--multi-dataset--your_tts.zip")
    if os.path.exists(corrupt_zip):
        # Check if actually extracted already
        model_files = [f for f in os.listdir(yourtts_dir) if f.endswith('.pth') or f.endswith('.json')]
        if not model_files:
            print("  Model zip appears corrupt, removing for re-download...")
            os.remove(corrupt_zip)
    
    yourtts = CoquiTTS("tts_models/multilingual/multi-dataset/your_tts").to(DEVICE)
    
    vram_used = torch.cuda.memory_allocated() / 1024**3
    print(f"✓ YourTTS loaded | VRAM: {vram_used:.2f} GB")

    print(f"\nSynthesizing {len(test_sentences)} sentences with YourTTS...\n")

    for si, sentence in enumerate(test_sentences):
        t0 = time.time()
        try:
            out_path = str(tier3_dir / f"yourtts_sent{si}.wav")
            yourtts.tts_to_file(
                text=sentence,
                language="en",
                speaker_wav=ref_paths[0],
                file_path=out_path,
            )
            elapsed = time.time() - t0
            
            wav_np, sr = librosa.load(out_path, sr=None)
            dur = len(wav_np) / sr
            rms = np.sqrt(np.mean(wav_np**2))
            expected_dur = len(sentence.split()) * 0.35
            dur_ratio = dur / expected_dur if expected_dur > 0 else 0
            
            yourtts_results.append({
                "sentence": sentence,
                "duration": dur,
                "expected_dur": expected_dur,
                "dur_ratio": dur_ratio,
                "rms": float(rms),
                "inference_time": elapsed,
                "sample_rate": sr,
                "path": out_path,
                "success": True,
            })
            
            status = "✓" if 0.5 < dur_ratio < 2.5 else "⚠️"
            print(f"[{si}] {status} dur={dur:.1f}s (ratio={dur_ratio:.2f}) | RMS={rms:.4f} | {elapsed:.2f}s")
            print(f"    \"{sentence[:60]}\"")
            display(Audio(out_path, rate=sr))
            
        except Exception as e:
            elapsed = time.time() - t0
            yourtts_results.append({"sentence": sentence, "success": False, "error": str(e)})
            print(f"[{si}] ✗ ERROR: {e}")
        
        torch.cuda.empty_cache()

    print(f"\n✓ YourTTS: {sum(1 for r in yourtts_results if r.get('success'))}/{len(test_sentences)} succeeded")

except Exception as e:
    print(f"⚠️ YourTTS failed to load: {e}")
    print("Skipping Tier 3 — F5-TTS from Tier 2 will be the winner.")
    yourtts_results = []

Freeing VRAM (unloading F5-TTS)...
✓ VRAM freed: 7.5 GB available

Loading YourTTS...
 > Downloading model to /home/nibiru/.local/share/tts/tts_models--multilingual--multi-dataset--your_tts


100%|█████████▉| 423M/425M [00:30<00:00, 29.6MiB/s] 

 > Model's license - CC BY-NC-ND 4.0
 > Check https://creativecommons.org/licenses/by-nc-nd/4.0/ for more info.
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Model fully restored. 
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:64
 | 

 > Text splitted to sentences.
['Hello, how are you doing today?', 'I hope everything is going well.']
 > Processing time: 0.5344846248626709
 > Real-time factor: 0.11357514340473243
[1] ✓ dur=4.7s (ratio=1.12) | RMS=0.1062 | 0.54s
    "Hello, how are you doing today? I hope everything is going w"


 > Text splitted to sentences.
['The president announced major reforms to the healthcare system yesterday.']
 > Processing time: 0.3715078830718994
 > Real-time factor: 0.06466629818483889
[2] ✓ dur=5.7s (ratio=1.64) | RMS=0.0858 | 0.38s
    "The president announced major reforms to the healthcare syst"


 > Text splitted to sentences.
['She walked through the garden, admiring the beautiful flowers in bloom.']
 > Processing time: 0.3892059326171875
 > Real-time factor: 0.09108493625489995
[3] ✓ dur=4.3s (ratio=1.11) | RMS=0.1009 | 0.39s
    "She walked through the garden, admiring the beautiful flower"



✓ YourTTS: 4/4 succeeded


100%|██████████| 425M/425M [00:45<00:00, 29.6MiB/s]

## A/B Comparison & Winner Selection

Compare all 3 tiers on:
1. **Duration Ratio** — how close to expected speech length (ideal: ~1.0)
2. **RMS Energy** — volume level (higher = more audible, not clipped)
3. **Inference Speed** — seconds per sentence
4. **Subjective Quality** — listen and compare

In [8]:
"""
Cell 9: A/B Comparison Table
"""

def summarize_tier(name, results):
    """Compute summary stats for a tier's results."""
    successful = [r for r in results if r.get("success")]
    if not successful:
        return {"name": name, "success_rate": 0, "avg_dur_ratio": 0, 
                "avg_rms": 0, "avg_speed": 0, "score": 999}
    
    avg_ratio = np.mean([r["dur_ratio"] for r in successful])
    avg_rms = np.mean([r["rms"] for r in successful])
    avg_speed = np.mean([r["inference_time"] for r in successful])
    
    # Composite score (lower = better):
    # - Duration ratio distance from 1.0 (weight: 3)
    # - Speed penalty (weight: 0.5) 
    # - Low RMS penalty (weight: 2)
    ratio_penalty = abs(avg_ratio - 1.0) * 3
    speed_penalty = avg_speed * 0.5
    rms_penalty = max(0, 0.02 - avg_rms) * 200  # penalty if RMS < 0.02
    score = ratio_penalty + speed_penalty + rms_penalty
    
    return {
        "name": name,
        "success_rate": len(successful) / len(results),
        "avg_dur_ratio": avg_ratio,
        "avg_rms": avg_rms,
        "avg_speed": avg_speed,
        "score": score,
    }

# Build comparison (XTTS results from previous run are hardcoded since kernel was restarted)
tiers = []

# XTTS-v2 best config results (from Tier 1 grid search - config 4: temp=0.5, rep_pen=2.0)
xtts_best_results = [
    {"sentence": test_sentences[0], "dur_ratio": 2.74, "rms": 0.0855, "inference_time": 1.69, "success": True},
    {"sentence": test_sentences[1], "dur_ratio": 2.23, "rms": 0.0984, "inference_time": 1.83, "success": True},
    {"sentence": test_sentences[2], "dur_ratio": 6.29, "rms": 0.0687, "inference_time": 4.66, "success": True},
    {"sentence": test_sentences[3], "dur_ratio": 5.04, "rms": 0.0804, "inference_time": 4.05, "success": True},
]
best_xtts_params = {"temperature": 0.5, "repetition_penalty": 2.0, "top_k": 50, "top_p": 0.85}
tiers.append(summarize_tier("XTTS-v2 (fixed)", xtts_best_results))

if f5_results:
    tiers.append(summarize_tier("F5-TTS", f5_results))
if yourtts_results:
    tiers.append(summarize_tier("YourTTS", yourtts_results))

# Display comparison table
print("=" * 90)
print("A/B COMPARISON: All 3 TTS Engines")
print("=" * 90)
print(f"{'Engine':<20} {'Success%':<10} {'Dur Ratio':<12} {'Avg RMS':<10} {'Avg Speed':<12} {'Score ↓':<10}")
print("-" * 90)
for t in sorted(tiers, key=lambda x: x["score"]):
    marker = " 🏆" if t["score"] == min(x["score"] for x in tiers) else ""
    print(f"{t['name']:<20} {t['success_rate']*100:<10.0f} {t['avg_dur_ratio']:<12.2f} "
          f"{t['avg_rms']:<10.4f} {t['avg_speed']:<12.2f} {t['score']:<10.2f}{marker}")

# Select winner
winner = min(tiers, key=lambda x: x["score"])
print(f"\n🏆 WINNER: {winner['name']} (score: {winner['score']:.2f})")

# Side-by-side listening for first sentence
print(f"\n{'='*60}")
print(f"Side-by-side: \"{test_sentences[0][:50]}...\"")
print(f"{'='*60}")

# F5-TTS
if f5_results and f5_results[0].get("success"):
    print(f"\n▶ F5-TTS:")
    display(Audio(f5_results[0]["path"], rate=f5_results[0].get("sample_rate", 24000)))

# YourTTS
if yourtts_results and yourtts_results[0].get("success"):
    print(f"\n▶ YourTTS:")
    display(Audio(yourtts_results[0]["path"], rate=yourtts_results[0].get("sample_rate", 16000)))

A/B COMPARISON: All 3 TTS Engines
Engine               Success%   Dur Ratio    Avg RMS    Avg Speed    Score ↓   
------------------------------------------------------------------------------------------
YourTTS              100        1.25         0.1008     0.45         0.98       🏆
F5-TTS               100        1.35         0.1042     2.58         2.34      
XTTS-v2 (fixed)      100        4.08         0.0832     3.06         10.75     

🏆 WINNER: YourTTS (score: 0.98)

Side-by-side: "The quick brown fox jumps over the lazy dog...."

▶ F5-TTS:



▶ YourTTS:


In [ ]:
"""
Cell 10: Save Best Config & Create Reusable TTS Engine Module
"""

# Save comparison results
comparison_data = {
    "test_sentences": test_sentences,
    "tiers": tiers,
    "winner": winner["name"],
    "xtts_best_params": best_xtts_params,
    "ref_audio_paths": ref_paths,
}

with open(RESULTS_DIR / "comparison_results.json", "w") as f:
    json.dump(comparison_data, f, indent=2, default=str)

print(f"✓ Results saved to {RESULTS_DIR / 'comparison_results.json'}")

# Create reusable TTS engine module
tts_engine_code = '''"""
tts_engine.py — Unified TTS Engine for TeluguVoiceBridge v2
Auto-generated by 07b_tts_rebuild.ipynb

Supports: F5-TTS, YourTTS, XTTS-v2 (fixed)
Default engine: {winner}

Benchmark Results:
  YourTTS:  dur_ratio=1.25, speed=0.45s/sent, VRAM=0.34GB  (WINNER)
  F5-TTS:   dur_ratio=1.35, speed=2.58s/sent, VRAM=0.69GB  (Runner-up)
  XTTS-v2:  dur_ratio=4.08, speed=3.06s/sent, VRAM=3.56GB  (Broken)
"""
import os, torch, numpy as np, soundfile as sf
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Best reference audio paths (LJSpeech, 8-10s clips)
DEFAULT_REF_PATHS = {ref_paths}

# Best XTTS-v2 parameters (from grid search -- still produces bad output)
XTTS_PARAMS = {xtts_params}


class TTSEngine:
    """Unified TTS engine with support for multiple backends."""
    
    def __init__(self, engine="{default_engine}", device=None):
        self.device = device or DEVICE
        self.engine_name = engine
        self.model = None
        self._load_engine(engine)
    
    def _load_engine(self, engine):
        if "your" in engine.lower():
            self._load_yourtts()
        elif "f5" in engine.lower():
            self._load_f5tts()
        elif "xtts" in engine.lower():
            self._load_xtts()
        else:
            raise ValueError(f"Unknown engine: {{engine}}. Use 'yourtts', 'f5tts', or 'xtts'")
    
    def _load_yourtts(self):
        from TTS.api import TTS as CoquiTTS
        self.model = CoquiTTS("tts_models/multilingual/multi-dataset/your_tts").to(self.device)
        self.engine_name = "yourtts"
        self.sample_rate = 16000
        print(f"YourTTS loaded on {{self.device}}")
    
    def _load_f5tts(self):
        from f5_tts.api import F5TTS
        self.model = F5TTS(device=self.device)
        self.engine_name = "f5tts"
        self.sample_rate = 24000
        # Load reference transcript for F5-TTS
        import pandas as pd
        base_dir = Path(DEFAULT_REF_PATHS[0]).parents[2]
        manifest = pd.read_csv(base_dir / "metadata" / "tts_manifest.csv")
        ref_name = Path(DEFAULT_REF_PATHS[0]).name.replace(".wav", "")
        match = manifest[manifest.audio_path.str.contains(ref_name)]
        self.ref_transcript = match.iloc[0]["transcript"] if len(match) > 0 else ""
        print(f"F5-TTS loaded on {{self.device}}")
    
    def _load_xtts(self):
        from TTS.tts.configs.xtts_config import XttsConfig
        from TTS.tts.models.xtts import Xtts
        
        config = XttsConfig()
        config.load_json(os.path.expanduser(
            "~/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/config.json"
        ))
        self.model = Xtts.init_from_config(config)
        self.model.load_checkpoint(
            config,
            checkpoint_dir=os.path.expanduser(
                "~/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/"
            ),
            eval=True, use_deepspeed=False
        )
        self.model = self.model.to(self.device)
        self.sample_rate = 24000
        
        # Pre-compute conditioning latents
        self.gpt_cond_latent, self.speaker_embedding = self.model.get_conditioning_latents(
            audio_path=DEFAULT_REF_PATHS,
            gpt_cond_len=12, gpt_cond_chunk_len=4,
            sound_norm_refs=True,
        )
        self.engine_name = "xtts"
        print(f"XTTS-v2 loaded on {{self.device}}")
    
    def synthesize(self, text, language="en", ref_audio=None):
        """Synthesize speech. Returns (wav_numpy, sample_rate)."""
        ref = ref_audio or DEFAULT_REF_PATHS[0]
        
        if self.engine_name == "yourtts":
            import tempfile, librosa
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                tmp_path = tmp.name
            self.model.tts_to_file(text=text, language=language,
                                   speaker_wav=ref, file_path=tmp_path)
            wav, sr = librosa.load(tmp_path, sr=None)
            os.unlink(tmp_path)
            return wav, sr
        
        elif self.engine_name == "f5tts":
            wav, sr, _ = self.model.infer(
                ref_file=ref,
                ref_text=self.ref_transcript,
                gen_text=text,
            )
            if isinstance(wav, torch.Tensor):
                wav = wav.cpu().numpy()
            return np.array(wav).squeeze(), sr
        
        elif self.engine_name == "xtts":
            with torch.no_grad():
                out = self.model.inference(
                    text=text, language=language,
                    gpt_cond_latent=self.gpt_cond_latent,
                    speaker_embedding=self.speaker_embedding,
                    **XTTS_PARAMS,
                    speed=1.0, enable_text_splitting=True,
                )
            return out["wav"], 24000
    
    def synthesize_to_file(self, text, output_path, language="en", ref_audio=None):
        """Synthesize and save to file."""
        wav, sr = self.synthesize(text, language, ref_audio)
        sf.write(output_path, wav, sr)
        return output_path


if __name__ == "__main__":
    engine = TTSEngine()
    wav, sr = engine.synthesize("Hello, this is a test of the text to speech engine.")
    sf.write("test_output.wav", wav, sr)
    print(f"Saved test_output.wav ({{len(wav)/sr:.1f}}s at {{sr}}Hz)")
'''.format(
    winner=winner["name"],
    ref_paths=repr(ref_paths),
    xtts_params=repr(best_xtts_params),
    default_engine="yourtts" if "your" in winner["name"].lower() else ("f5tts" if "f5" in winner["name"].lower() else "xtts"),
)

engine_path = BASE_DIR / "tts_engine.py"
with open(engine_path, "w") as f:
    f.write(tts_engine_code)

print(f"✓ TTS engine module saved to {engine_path}")
print(f"  Default engine: {winner['name']}")
print(f"  Usage: from tts_engine import TTSEngine; engine = TTSEngine()")

# Cleanup VRAM
try:
    del yourtts
except NameError:
    pass
torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"  TTS REBUILD COMPLETE")
print(f"  Winner: {winner['name']}")
print(f"  All outputs in: {RESULTS_DIR}")
print(f"  Reusable module: {engine_path}")
print(f"{'='*60}")

✓ Results saved to /home/nibiru/Documents/sem6project/Speech2/pipeline_v2/results/tts_rebuild/comparison_results.json
✓ TTS engine module saved to /home/nibiru/Documents/sem6project/Speech2/pipeline_v2/tts_engine.py
  Default engine: YourTTS
  Usage: from tts_engine import TTSEngine; engine = TTSEngine()

  TTS REBUILD COMPLETE
  Winner: YourTTS
  All outputs in: /home/nibiru/Documents/sem6project/Speech2/pipeline_v2/results/tts_rebuild
  Reusable module: /home/nibiru/Documents/sem6project/Speech2/pipeline_v2/tts_engine.py


: 